# 페르소나와 과업 제약

페르소나 이름만 주는 것과 수행할 일·금지 사항·출력 형식을 함께 주는 것은 어떤 차이를 만들까요?

## 패키지 준비

첫 실행에서 수업 기준 버전을 설치합니다.

In [1]:
# 현재 커널에 수업 기준 패키지 버전을 설치합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## API 키와 모델 준비

상위 폴더의 `.env`를 찾고 키의 존재만 확인합니다.

In [2]:
# find_dotenv는 실행 폴더부터 상위 폴더의 .env를 찾고 키 값은 출력하지 않습니다.
import json, os, time
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model

env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("프로젝트 루트의 .env에 OPENAI_API_KEY가 필요합니다.")
model = init_chat_model("openai:gpt-5.4-mini", max_tokens=220)

## 고정 과제

모든 절에서 같은 고객 문의와 확인 단어를 사용합니다.

In [3]:
# TASK는 모든 비교에서 바꾸지 않는 합성 문의이고 MENTIONS는 품질 점수가 아닙니다.
TASK = """고객 문의: 어제 받은 텀블러 뚜껑이 깨졌습니다. 주문번호는 A-1042입니다.
새 상품으로 바꾸고 싶습니다. 회사 규정상 교환 접수에는 파손 사진이 필요합니다.
답변에는 분류, 필요한 다음 행동, 고객에게 보낼 한 문장을 포함하세요."""
MENTIONS = ["교환", "A-1042", "사진"]

## 측정 함수

단어 포함 표시는 원문 정독을 돕는 표시이며 품질 점수가 아닙니다.

`run(messages)`는 이 노트북에서 작성하는 보조 함수입니다. 역할·내용 목록을 받아 `model.invoke()`를 한 번 실행하고, 답 문자열·사용량·경과 초를 담은 딕셔너리를 반환합니다. `def` 셀은 함수를 정의할 뿐이며 뒤에서 `run(...)`을 호출할 때 모델 요청이 시작됩니다. `mention_flags(text)`는 지정 단어가 문자열에 있는지 표시한 딕셔너리를 반환합니다. 답의 뜻을 판단하는 모델 호출은 포함하지 않습니다.

In [4]:
# 모델 응답의 원문, 사용량, 경과 시간을 한 레코드로 보존합니다.
def run(messages):
    started = time.perf_counter()
    answer = model.invoke(messages)
    return {"text": answer.text, "usage": answer.usage_metadata,
            "latency_s": round(time.perf_counter() - started, 3)}

def mention_flags(text):
    return {word: word in text for word in MENTIONS}

## 페르소나만 지정

친절한 상담원이라는 성격만 지정합니다.

페르소나는 답하는 사람의 역할·말투·관점을 지정한 설명입니다. 아래 첫 조건의 “친절한 고객 상담원”은 태도에 관한 문장이고, 다음 조건의 세 줄 형식과 규정 밖 약속 금지는 수행 조건입니다. 두 번째 조건이 더 길다는 사실 자체를 품질 개선으로 해석하지 않습니다.

회의록 정리에서는 “숙련된 비서”라는 역할에 결정 사항·담당자·기한을 추출하라는 과업과 미정 기한을 추측하지 말라는 조건을 따로 붙일 수 있습니다. 같은 구분을 고객 문의에 적용해 말투, 필수 정보, 금지 사항, 출력 배열을 각각 읽습니다.

`persona`와 `bounded`는 함수가 아니라 `run()`이 반환한 결과 딕셔너리입니다. `result["text"]`는 모델이 생성한 원문이고 `len(...)`은 그 문자열의 글자 수입니다. 요청에 쓴 글자 수나 토큰 수와 구분합니다.

In [5]:
# 페르소나만 둔 기준 응답의 원문과 사용량을 보존합니다.
persona = run([("system", "당신은 친절한 고객 상담원입니다."), ("user", TASK)])
print(json.dumps(persona, ensure_ascii=False, indent=2))

{
  "text": "**분류:** 교환 요청 / 파손 상품 접수\n\n**필요한 다음 행동:**  \n파손된 뚜껑 사진을 첨부해 주시면 교환 접수를 진행할 수 있습니다. 주문번호 **A-1042** 기준으로 확인하겠습니다. 사진 확인 후 새 상품으로 교환 절차를 안내드리겠습니다.\n\n**고객에게 보낼 한 문장:**  \n파손된 뚜껑 사진을 보내주시면 주문번호 A-1042로 확인 후 새 상품 교환을 바로 도와드리겠습니다.",
  "usage": {
    "input_tokens": 96,
    "output_tokens": 125,
    "total_tokens": 221,
    "input_token_details": {
      "audio": 0,
      "cache_read": 0
    },
    "output_token_details": {
      "audio": 0,
      "reasoning": 0
    }
  },
  "latency_s": 1.287
}


## 과업과 제약을 함께 지정

누구인지에 더해 무엇을 어떤 범위에서 답할지 적습니다.

In [6]:
# 같은 페르소나에 세 줄 형식과 금지 조건을 추가합니다.
bounded = run([
    ("system", "당신은 친절한 고객 상담원입니다."),
    ("user", "세 줄로 답하세요. 1) 분류 2) 다음 행동 3) 고객 문장. "
             "규정에 없는 환불이나 보상을 약속하지 마세요.\n" + TASK),
])
print(json.dumps(bounded, ensure_ascii=False, indent=2))

{
  "text": "1) 분류: 배송 후 파손으로 인한 교환 문의입니다.  \n2) 다음 행동: 주문번호 A-1042와 함께 파손된 뚜껑 사진을 보내주시면 교환 접수를 진행하겠습니다.  \n3) 고객 문장: 어제 받으신 텀블러 뚜껑 파손 건은 확인되었으며, 교환 접수를 위해 주문번호 A-1042와 파손 사진을 보내주시면 안내드리겠습니다.",
  "usage": {
    "input_tokens": 134,
    "output_tokens": 109,
    "total_tokens": 243,
    "input_token_details": {
      "audio": 0,
      "cache_read": 0
    },
    "output_token_details": {
      "audio": 0,
      "reasoning": 0
    }
  },
  "latency_s": 1.235
}


## 결과 비교

같은 세 평가 항목과 응답 길이를 함께 확인합니다.

In [7]:
# 페르소나 조건과 구체 제약 조건의 글자 수·토큰·지연을 원문 이름과 연결합니다.
for name, result in {"persona": persona, "task_constraints": bounded}.items():
    print(name, mention_flags(result["text"]), "글자 수", len(result["text"]),
          "토큰", result["usage"], "지연", result["latency_s"])

persona {'교환': True, 'A-1042': True, '사진': True} 글자 수 220 토큰 {'input_tokens': 96, 'output_tokens': 125, 'total_tokens': 221, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}} 지연 1.287
task_constraints {'교환': True, 'A-1042': True, '사진': True} 글자 수 176 토큰 {'input_tokens': 134, 'output_tokens': 109, 'total_tokens': 243, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}} 지연 1.235
